# smFRET calibration: light-path prior → data-optimized posterior

Accurate FRET needs calibration factors — the detection/quantum-yield ratio `gamma`,
donor leakage `alpha`, direct acceptor excitation `delta`, backgrounds and the Förster
radius `R0`. In ChiSurf these are **fitting parameters** whose

* **prior** comes from the **light-path calculator** (the optics model), and
* **posterior** comes from **optimizing against the measured data**.

The light-path value is only the prior; the data refines it. This notebook walks the
workflow on simulated data with a known ground truth.


## 1. Light-path prior

The light-path calculator produces spectral crosstalk matrices. From them we compute
`gamma`/`alpha`/`delta` and attach them as **Gaussian priors** to a `CalibrationParameters`
group. (Here we use a small synthetic light-path payload; in practice it comes from
`LightPathSimulator.get_crosstalk_matrices()`.)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from chisurf.core.fluorescence.burst.es import apparent_es, corrected_es

from chisurf.core.fluorescence.fret.calibration import (
    CalibrationParameters,
    global_es_correction,
    refine_calibration,
    set_priors_from_lightpath,
)

# A synthetic light-path payload (excitation: laser x dye; emission: dye x detector).
# cGD=0.74 -> gamma prior ~ 1/0.74 = 1.35; leakage/direct-excitation small.
matrices = {
    "excitation": {"rows": ["532"], "columns": ["donor", "acceptor"], "values": [[1.0, 0.05]]},
    "emission": {
        "rows": ["donor", "acceptor"],
        "columns": ["green", "red"],
        "values": [[0.74, 0.06], [0.0, 1.0]],
    },
}
calib = CalibrationParameters()
factors = set_priors_from_lightpath(calib, matrices, "donor", "acceptor", "green", "red", r0=53.0)
print("light-path factors (prior means):", {k: round(v, 3) for k, v in factors.items()})
print(
    "gamma prior:",
    calib._gamma.prior.__class__.__name__,
    f"mu={calib._gamma.prior.mu:.3f} sigma={calib._gamma.prior.sigma:.3f}",
)

## 2. Simulated data with a known calibration

We synthesize three FRET populations (E = 0.25, 0.55, 0.80) measured with a **true**
`gamma=1.4`, leakage `alpha=0.08` and direct excitation `delta=0.05`. These differ from
the light-path prior — the point is that the data will pull the estimate toward the truth.


In [ ]:
rng = np.random.default_rng(0)
gamma_true, alpha_true, delta_true = 1.4, 0.08, 0.05
E_true = [0.25, 0.55, 0.80]
green, red, yellow, labels = [], [], [], []
for i, E in enumerate(E_true):
    tot = rng.poisson(200, 500).astype(float)
    f_dd = (1 - E) * tot
    f_aa = rng.poisson(200, 500).astype(float)
    ida = gamma_true * E * tot + alpha_true * f_dd + delta_true * f_aa
    green.append(rng.poisson(np.clip(f_dd, 0, None)))
    red.append(rng.poisson(np.clip(ida, 0, None)))
    yellow.append(rng.poisson(np.clip(f_aa, 0, None)))
    labels.append(np.full(tot.size, i))
green, red, yellow, labels = (np.concatenate(x).astype(float) for x in (green, red, yellow, labels))

app = apparent_es(green, red, yellow)
print("apparent E per population:", [round(app["E"][labels == i].mean(), 3) for i in range(3)])

## 3. Layered calibration: global E-S estimate → prior-regularized posterior

`global_es_correction` recovers `gamma` from the E-S population plot (Lee 2005 / Hellenkamp
2018). `refine_calibration` then combines that **data estimate** with the **light-path prior**
by precision weighting — strong data follows the data, weak data falls back to the prior.


In [ ]:
est = global_es_correction(green, red, yellow, labels, alpha=alpha_true, delta=delta_true)
print("data gamma (E-S fit): {:.3f}   (true {:.2f})".format(est["gamma"], gamma_true))

calib.alpha, calib.delta = alpha_true, delta_true  # from donor-only/acceptor-only or the prior
post = refine_calibration(calib, green, red, yellow, labels)
print(
    "posterior gamma: {:.3f}   (prior {:.3f}, data {:.3f}, data_sigma {:.3f})".format(
        post["gamma"], post["gamma_prior"], post["gamma_data"], post["data_sigma"]
    )
)

## 4. Accurate FRET after calibration

Correcting the counts with the posterior calibration recovers the true efficiencies,
while the uncorrected proximity ratio is biased.


In [ ]:
corr = corrected_es(green, red, yellow, gamma=post["gamma"], alpha=calib.alpha, delta=calib.delta)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for i, c in zip(range(3), ("C0", "C1", "C2")):
    ax[0].hist(app["E"][labels == i], bins=40, range=(0, 1), alpha=0.6, color=c)
    ax[1].hist(corr["E"][labels == i], bins=40, range=(0, 1), alpha=0.6, color=c)
for a, t in zip(ax, ("apparent (uncorrected)", "corrected")):
    for E in E_true:
        a.axvline(E, color="k", ls="--", lw=0.8)
    a.set_title(t)
    a.set_xlabel("FRET efficiency E")
ax[0].set_ylabel("bursts")
fig.tight_layout()
print(
    "corrected E per population:",
    [round(corr["E"][labels == i].mean(), 3) for i in range(3)],
    "   true",
    E_true,
)

## 5. Real tttrlib photon simulation (optional)

The same correction works end-to-end on photons simulated with tttrlib and selected as
bursts. We bake a known `gamma` into the simulation (`BurstWorkflow.simulate(gamma=...)`)
and recover the true efficiencies from the burst proximity ratios. (Requires the tttrlib
`SimEngine` build and MMFDB; skipped gracefully otherwise.)


In [ ]:
try:
    import tempfile

    from chisurf.plugins.burst.burst_analysis.api import BurstWorkflow

    wf = BurstWorkflow.demo(workdir=tempfile.mkdtemp())
    sim = wf.simulate(fret=(0.25, 0.75), exchange_rate=0.0, gamma=1.6, n_photons=400_000, seed=7)
    bursts = wf.select_bursts(sim.handle, setup=sim.setup, min_photons=40)
    pr = bursts.bva("green", "red").table["Proximity Ratio Mean"].to_numpy(float)
    pr = pr[np.isfinite(pr)]
    e_corr = corrected_es(1 - pr, pr, gamma=sim.truth.gamma)["E"]
    lo = pr < 0.5
    print("true gamma:", sim.truth.gamma)
    print(
        f"corrected E (low/high): {e_corr[lo].mean():.3f} / {e_corr[~lo].mean():.3f}   true 0.25 / 0.75"
    )
    wf.close()
except Exception as exc:
    print("tttrlib/MMFDB simulation unavailable:", exc)

## Summary

* Calibration factors are `FittingParameter`s; the **light-path calculator sets their prior**.
* `global_es_correction` gives the **data** estimate of `gamma`; `refine_calibration` returns the
  **posterior** (precision-weighted with the prior).
* Correcting with the posterior yields **accurate FRET**; the same factors can feed ndxplorer.

See `okf/references/fret-calibration.md` for the design and the remaining phases
(session-shared calibration for global analysis, the ndx bridge).
